<h1> <font color="blue"> <center> Logistic Reegression and GIS </h1>
06 October 2025
<h1> <font color="Red"> <center> Abdul Rehman</h1>

In [50]:
# Import Libraries
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import (confusion_matrix,classification_report,roc_curve,roc_auc_score,precision_recall_curve)
from matplotlib import pyplot as plt
import geopandas as gpd
import numpy as np

In [51]:
path = '/home/fortbend/Assignment_4/Abdul'
fname = 'NBI2024LR.csv'
os.chdir(path)

In [52]:
a= pd.read_csv(fname)
a.head()

,FID,LATDD,LONDD,AGE,RECON,ADT,MATL,UNSAFE
0,35185702101560762,35.315839,-101.935450,16.0,0,100,1,0
1,29362100094272880,29.605833,-94.458000,19.0,0,1,5,0
2,29362940094254920,29.608167,-94.430333,19.0,0,1,5,0
3,29365160094324020,29.614333,-94.544500,12.0,0,100,5,0
4,29413300094044860,29.692500,-94.080167,17.0,1,80,7,1


In [53]:
a['AGE2'] = np. power(a.AGE,2)
a.head()

,FID,LATDD,LONDD,AGE,RECON,ADT,MATL,UNSAFE,AGE2
0,35185702101560762,35.315839,-101.935450,16.0,0,100,1,0,256.0
1,29362100094272880,29.605833,-94.458000,19.0,0,1,5,0,361.0
2,29362940094254920,29.608167,-94.430333,19.0,0,1,5,0,361.0
3,29365160094324020,29.614333,-94.544500,12.0,0,100,5,0,144.0
4,29413300094044860,29.692500,-94.080167,17.0,1,80,7,1,289.0


In [82]:
X.describe()

,AGE,AGE2,ADT,MATL,RECON
count,56514.000000,56514.000000,56514.000000,56514.000000,56514.000000
mean,37.289397,1892.952773,10830.846604,2.729518,0.194571
std,22.415676,1919.849727,23251.193571,1.887833,0.395874
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,18.000000,324.000000,397.000000,1.000000,0.000000
50%,35.000000,1225.000000,2700.500000,1.000000,0.000000
75%,55.000000,3025.000000,10621.500000,5.000000,0.000000
max,124.000000,15376.000000,778093.000000,9.000000,1.000000


In [60]:
a.dropna(inplace=True)

In [62]:

xcol = ['AGE','AGE2','ADT','MATL','RECON']
X= a.copy()
X= X.loc[:,xcol]
Y= a.copy()
Y= Y.loc[:,'UNSAFE']

In [63]:
# Train.Test Split
xtrain, xtest, ytrain, ytest = train_test_split(X, Y, test_size=0.3, stratify=Y, random_state=10)

In [64]:
model = LR(class_weight='balanced', random_state=20, solver='liblinear')
model.fit(xtrain, ytrain)

LogisticRegression(class_weight='balanced', random_state=20, solver='liblinear')

In [65]:
model.score(xtrain,ytrain)

0.6268864228114968

In [67]:
yscores = model.predict_proba(xtest)[:,1]
yscores

array([0.72946878, 0.34127691, 0.2833685 , ..., 0.12808133, 0.17290848,
       0.4593698 ])

In [69]:
# ROC Curve and area under the curve(auc)
fpr,tpr,thresh = roc_curve(ytest,yscores)
fpr,tpr,thresh

(array([0.00000000e+00, 0.00000000e+00, 1.28924128e-04, ...,
        9.99677690e-01, 9.99871076e-01, 1.00000000e+00]),
 array([0.00000000e+00, 6.93481276e-04, 6.93481276e-04, ...,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00]),
 array([       inf, 0.95728473, 0.94560766, ..., 0.03846645, 0.03779835,
        0.03706593]))

In [75]:
auc= roc_auc_score(ytest,yscores)
print('AUC:', auc)

AUC: 0.7284441003487477


In [77]:
# Optimal Cutoff
J = tpr-fpr
ocutoff= np.argmax(J)
ocutoff = thresh[ocutoff]
ocutoff

0.3802771767620956

In [79]:
# Create a confusion matrix with optimal threshold
unsafe = (yscores>=ocutoff).astype(int)
confusion_matrix(ytest,unsafe)

array([[7453, 8060],
       [ 224, 1218]])

In [80]:
sum(unsafe)

9278